In [2]:
# -*- coding: utf-8 -*-
"""
페르소나 JSON + 제품 CSV -> '월구매 예측' 프롬프트 자동 생성 스크립트
- 입력:
  - PERSONA_JSON: 2200명 페르소나 (예: /mnt/data/persona_core_summary_fixed (1).json)
  - PRODUCTS_CSV: 신제품 정보 (예: /mnt/data/product_info_v2 (1).csv)
- 출력:
  - build_monthly_prompt(persona, products, target_month=...) -> str
  - (옵션) batch로 모든 페르소나 프롬프트 파일 저장
"""

import json
import re
import math
import pandas as pd
from datetime import datetime
from typing import List, Dict, Any
from textwrap import dedent

# ===== 0. 경로 설정 =====
PERSONA_JSON = "persona.json"
PRODUCTS_CSV = "product_info.csv"  # 사용자 경로에 맞게 변경

# ===== 1. 유틸 =====
def _safe_get(d: dict, path: List[str], default=None):
    cur = d
    for k in path:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(k)
        if cur is None:
            return default
    return cur

def _to_float(x, default=None):
    try:
        if x is None or (isinstance(x, float) and math.isnan(x)):
            return default
        return float(str(x).replace(",", "").strip())
    except:
        return default

def _norm_1to9(x, default=0.5):
    """1~9 점수를 0~1로 정규화. 소수/결측 방어."""
    v = _to_float(x, None)
    if v is None:
        return default
    v = max(1.0, min(9.0, v))
    return (v - 1.0) / 8.0

# ===== 2. 구매주기 텍스트 -> 월간 빈도 추정 =====
_PURCHASE_PATTERNS = [
    (re.compile(r"주\s*(\d+)\s*~\s*(\d+)\s*회"), lambda a,b: ((int(a)+int(b))/2.0) * 4.3),
    (re.compile(r"주\s*(\d+)\s*회"), lambda a: int(a)*4.3),
    (re.compile(r"(\d+)\s*주\s*일?\s*에\s*1\s*회"), lambda a: 4.3/float(a)),
    (re.compile(r"(\d+)\s*주\s*\/?\s*1\s*회"), lambda a: 4.3/float(a)),
    (re.compile(r"(?:1|한)\s*달\s*에\s*1\s*회"), lambda : 1.0),
    (re.compile(r"월\s*(\d+)\s*회"), lambda a: float(a)),
    (re.compile(r"분기\s*1\s*회"), lambda : 1.0/3.0),
]

def purchase_cycle_to_monthly_freq(text: str, fallback: float=1.0) -> float:
    if not text:
        return fallback
    s = str(text).strip()
    for pat, fn in _PURCHASE_PATTERNS:
        m = pat.search(s)
        if m:
            try:
                return float(fn(*m.groups()))
            except:
                continue
    if "가끔" in s or "드묾" in s:
        return 0.5
    if "자주" in s:
        return 2.0
    return fallback

# ===== 3. 제품 데이터 로드 & 스키마 보강 =====
def load_products(path: str) -> List[Dict[str, Any]]:
    df = pd.read_csv(path)

    size_ml = []
    for name in df["product_name"].astype(str):
        m = re.search(r"(\d+)\s*(g|ml|mL|G|ML)", name)
        size_ml.append(m.group(1)+m.group(2).lower() if m else "")
    df["size"] = size_ml

    for col in ["price_avg","launch_date","packaging","promotion_flag","ad_channel","premium_flag","competitor_tag","currency"]:
        if col not in df.columns:
            df[col] = None
    df["currency"] = df["currency"].fillna("KRW")

    def parse_features(x: str) -> List[str]:
        if not isinstance(x, str): return []
        parts = re.split(r"[,/;]\s*", x)
        cleaned = []
        for p in parts:
            p = p.strip()
            p = re.sub(r"[^0-9a-zA-Z가-힣()+\- ]", "", p)
            if p:
                cleaned.append(p[:40])
        return cleaned[:8]
    df["features_structured"] = df["product_feature"].apply(parse_features)

    products = []
    for _, r in df.iterrows():
        products.append({
            "name": r["product_name"],
            "category_level_1": r["category_level_1"],
            "category_level_2": r["category_level_2"],
            "category_level_3": r["category_level_3"],
            "size": r["size"] or "",
            "price": _to_float(r["price_avg"], None),
            "currency": r["currency"] or "KRW",
            "launch_date": str(r["launch_date"]) if pd.notna(r["launch_date"]) else None,
            "packaging": r["packaging"] if pd.notna(r["packaging"]) else None,
            "promotion_flag": bool(r["promotion_flag"]) if pd.notna(r["promotion_flag"]) else None,
            "ad_channel": r["ad_channel"] if pd.notna(r["ad_channel"]) else None,
            "premium_flag": r["premium_flag"] if pd.notna(r["premium_flag"]) else None,
            "competitor_tag": r["competitor_tag"] if pd.notna(r["competitor_tag"]) else None,
            "features_structured": r["features_structured"],
            "raw_feature": r["product_feature"],
        })
    return products

# ===== 4. 페르소나 로드 =====
def load_personas(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict) and "personas" in data:
        return data["personas"]
    if isinstance(data, list):
        return data
    raise ValueError("지원하지 않는 JSON 구조입니다. 최상위가 list 또는 {'personas': [...]} 이어야 합니다.")

# ===== 5. 페르소나 요약 =====
def summarize_persona(p: dict) -> str:
    persona_id = p.get("persona_id", p.get("id", "N/A"))
    seg_id = p.get("segment_id","")
    seg_label = p.get("segment_label","")
    demo = _safe_get(p, ["demographics"], {}) or {}
    shop = _safe_get(p, ["shopping_profile"], {}) or {}
    ori  = _safe_get(p, ["orientations"], {}) or {}

    demo_txt = ", ".join(f"{k}:{v}" for k,v in demo.items() if v not in [None,""])

    # 리스트 None 방지
    top_categories = shop.get("top_categories") or []
    top_categories_str_list = [str(item) for item in top_categories if item is not None]

    dec_criteria = shop.get("decision_criteria") or []
    dec_criteria_str_list = [str(item) for item in dec_criteria if item is not None]

    pc = shop.get("purchase_cycle") or ""
    ori_txt = ", ".join(f"{k}:{v}" for k,v in ori.items() if v not in [None,""])

    lines = [
        f"persona_id: {persona_id}",
        f"segment_id: {seg_id} / segment_label: {seg_label}",
        f"demographics: {demo_txt}" if demo_txt else "demographics: N/A",
        f"purchase_cycle: {pc}" if pc else "purchase_cycle: N/A",
        f"top_categories: {', '.join(top_categories_str_list) if top_categories_str_list else 'N/A'}",
        f"decision_criteria: {', '.join(dec_criteria_str_list) if dec_criteria_str_list else 'N/A'}",
        f"orientations(1-9): {ori_txt}" if ori_txt else "orientations: N/A",
    ]
    return "\n".join(lines)

def risk_notes(p: dict) -> List[str]:
    ori  = _safe_get(p, ["orientations"], {}) or {}
    notes = []
    if _to_float(ori.get("brand_loyalty"), 0) >= 7:
        notes.append("브랜드 충성 높음 → 기존 익숙한 SKU 고집으로 신제품 전환 낮음(프로모션 시 완화)")
    if _to_float(ori.get("price_sensitivity"), 0) >= 7:
        notes.append("가격 민감 높음 → 평균가 대비 가격상승에 민감(프로모션/대용량에 반응)")
    if _to_float(ori.get("hmr"), 0) >= 7:
        notes.append("HMR 성향 높음 → 간편식/즉석조리 카테고리 적합도↑")
    if _to_float(ori.get("health"), 0) >= 7:
        notes.append("건강/영양 성향 높음 → 저첨가/클린라벨/고단백/저나트륨 속성 선호")
    return notes or ["명시적 리스크 없음(기본 검증 규칙 유지)"]

# ===== 6. 제품 블록 =====
def products_block(products: List[Dict[str,Any]]) -> str:
    rows = []
    for i, prd in enumerate(products):
        rows.append(
            f"- {i+1}. {prd['name']} ({prd.get('size','')}) | "
            f"카테고리:{prd.get('category_level_1','')}/{prd.get('category_level_2','')}/{prd.get('category_level_3','')} | "
            f"예상소비자가:{prd.get('price','N/A')} {prd.get('currency','KRW')} | "
            f"프리미엄:{prd.get('premium_flag','N/A')} | 프로모션:{prd.get('promotion_flag','N/A')}"
        )
    return "\n".join(rows)

# ===== 7. (신규) '상시 월구매' 프롬프트 생성 =====
def build_monthly_prompt_generic(
    persona: dict,
    products: List[Dict[str,Any]],
    market_context: str = "대한민국 가공식품 소매시장(온·오프라인 합산)",
    base_rate_hint: str = "월간 구매확률은 0~1의 현실적 범위 내에서 제한되며, 특정 월/연도의 이벤트는 고려하지 않는다(상시 평균 가정).",
    cap_prob_low: float = 0.02,
    cap_prob_high: float = 0.60
) -> str:
    # 페르소나 요약/리스크
    persona_summary = summarize_persona(persona)
    risks = risk_notes(persona)

    # 월빈도 추정 힌트
    pc_text = _safe_get(persona, ["shopping_profile","purchase_cycle"], "")
    monthly_freq_est = purchase_cycle_to_monthly_freq(pc_text, fallback=1.0)

    # 성향 정규화 값(0~1)
    ori = _safe_get(persona, ["orientations"], {}) or {}
    ori_norm = {
        "health": _norm_1to9(ori.get("health")),
        "price_sensitivity": _norm_1to9(ori.get("price_sensitivity")),
        "hmr": _norm_1to9(ori.get("hmr")),
        "brand_loyalty": _norm_1to9(ori.get("brand_loyalty")),
        "premium": _norm_1to9(ori.get("premium")),
        "convenience": _norm_1to9(ori.get("convenience")),
        "variety_seeking": _norm_1to9(ori.get("variety_seeking")),
    }

    prompt = dedent(f"""
    # 역할
    당신은 **소비자 페르소나 기반 월간 구매예측 애널리스트**입니다. 내부 사고는 메모로만 사용하고, 최종 출력은 지정된 JSON 포맷으로만 반환합니다.

    # 과업
    아래 **페르소나**에 대해, 특정 연/월을 가정하지 않고 **상시 평균적인 '한 달'** 동안 다음 **제품 후보** 각각을 구매할 **구매확률(0~1)**과 **예상 구매수량(개)**을 추정하세요.
    예산·채널·빈도 제약, 대체·보완 관계를 반영하되, **명절·계절·프로모션 등 특정 시점 이벤트는 고려하지 않습니다.**

    # 페르소나(요약)
    {persona_summary}

    # 전처리 힌트(모델 내부 메모용)
    - purchase_cycle 월빈도 추정값: ~{monthly_freq_est:.2f}/월
    - orientations 정규화(0~1): {ori_norm}

    # 리스크·편향 유의
    {', '.join(risks)}

    # 시장 컨텍스트
    {market_context}
    - 베이스레이트 힌트: {base_rate_hint}

    # 제품 후보
    {products_block(products)}

    # 모델링 규칙
    1) **예산/빈도 제약**: 상시 월 예산과 기존 소비주기 준수. 총지출이 비현실적으로 커지지 않도록 수량/확률 동시 조정.
    2) **대체·보완**: 유사 카테고리 간 경쟁/보완성 반영(동시구매/대체 확률).
    3) **교정(Calibration)**: 구매확률을 {cap_prob_low}~{cap_prob_high} 범위 중심으로 유도(0/1 극단 회피).
    4) **검증**: 총지출·최대지출·월빈도·재고누적(가정) 체크 후 이상치 수정.
    5) **시간성 배제**: 특정 연/월, 시즌·프로모션 등 시점 요인 반영 금지.

    # 출력 포맷(JSON만 반환, 추가 텍스트 금지)
    {{
      "persona_id": {persona.get('persona_id', persona.get('id', 'null'))},
      "assumptions": {{
        "seasonality": "시간적 요인 배제(상시 평균 가정)",
        "promotion": "시간적 요인 배제(상시 평균 가정)",
        "substitution_rules": "문장",
        "budget_binding": true/false
      }},
      "forecast": [
        {{
          "product": "제품명",
          "persona_fit_score": 0.0,
          "purchase_prob": 0.0,
          "expected_qty": 0,
          "expected_spend_KRW": 0,
          "drivers": ["최대 3개(상시 요인)"],
          "barriers": ["최대 3개(상시 요인)"]
        }}
        "... 모든 제품 반복 ..."
      ],
      "sanity_checks": {{
        "total_monthly_spend_KRW": 0,
        "max_single_item_spend_KRW": 0,
        "exceeds_budget": true/false,
        "notes": ["조정/수정 근거(상시 가정)"]
      }}
    }}

    # 절차
    - (내부 메모) 제약/가정 정리 → 초기치 → 베이스레이트/예산 교정 → 대체 반영 → 검증/수정.
    - 최종 응답에는 **JSON만** 포함하세요(설명 금지).
    """).strip()

    return prompt
    
# ===== 8. 실행 예시 / 배치 생성 =====
# 데이터 로드
try:
    personas = load_personas(PERSONA_JSON)
except Exception as e:
    print(f"페르소나 파일('{PERSONA_JSON}') 로드 오류:", e)
    personas = []

try:
    products = load_products(PRODUCTS_CSV)
except Exception as e:
    print(f"제품 파일('{PRODUCTS_CSV}') 로드 오류:", e)
    products = []

# 데모: 앞의 1명에 대해 '상시 월구매' 프롬프트 출력
if personas and products:
    demo_prompt = build_monthly_prompt_generic(personas[0], products)
    print("\n===== DEMO PROMPT (상시 월구매, 1명) =====\n")
    print(demo_prompt)
else:
    print("\n데이터 파일 로드에 실패하여 프롬프트를 생성할 수 없습니다. 파일 경로를 확인해주세요.")

# (옵션) 전체 배치 저장 — 상시 월구매 기준 JSONL
if personas and products:
    out_path = f"prompts_monthly_generic_{datetime.now().strftime('%Y%m%d_%H%M%S')}.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for p in personas:
            prompt = build_monthly_prompt_generic(p, products)
            rec = {"persona_id": p.get("persona_id", p.get("id")), "prompt": prompt}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print("\n===== BATCH SAVE COMPLETE (MONTHLY - GENERIC) =====")
    print("저장 완료:", out_path)


===== DEMO PROMPT (상시 월구매, 1명) =====

# 역할
    당신은 **소비자 페르소나 기반 월간 구매예측 애널리스트**입니다. 내부 사고는 메모로만 사용하고, 최종 출력은 지정된 JSON 포맷으로만 반환합니다.

    # 과업
    아래 **페르소나**에 대해, 특정 연/월을 가정하지 않고 **상시 평균적인 '한 달'** 동안 다음 **제품 후보** 각각을 구매할 **구매확률(0~1)**과 **예상 구매수량(개)**을 추정하세요.
    예산·채널·빈도 제약, 대체·보완 관계를 반영하되, **명절·계절·프로모션 등 특정 시점 이벤트는 고려하지 않습니다.**

    # 페르소나(요약)
    persona_id: 1
segment_id: 2 / segment_label: 프리미엄+편의성선호형
demographics: age:52, gender:여자, region:부산, household:1인 가구, income_bracket:100-200만원 미만, job:단순노무 종사자, marital_status:미혼(사별/이혼 포함), dual_income:False
purchase_cycle: 2주일에 1회
top_categories: 간편식(HMR, 즉석조리, 즉석섭취, 밀키트, 신선편의, 커피 및 차(커피, 커피음료, 잎차, 티백, 곡물차 등
decision_criteria: 품질, 안전성
orientations(1-9): health:7, price_sensitivity:6, hmr:5, brand_loyalty:9, premium:7, convenience:8, variety_seeking:6

    # 전처리 힌트(모델 내부 메모용)
    - purchase_cycle 월빈도 추정값: ~2.15/월
    - orientations 정규화(0~1): {'health': 0.75, 'price_sensitivity': 0.625, 'hmr': 0.5, 'brand_loyalty': 1.0, 'premium': 0.75, 'con